# 03 · Explain the risk

Which factors raise or lower a risk score, using SHAP on the final model of a chosen
run (trained on every row with a known outcome), and the scores, risk bands and top
factors for what is still open. This is what the officials will see in the M4
dashboard; the decision stays with them.

> The tables hold member codes and features only: no names, no free text (decision D-016). The loan-default label is **provisional** until the Chama agrees it (D-018): `defaulted` = rolled over while unpaid, overdue (confirmed), or lost track.

In [ ]:
# Setup: runs in Google Colab and on this computer.
# Colab, once: add a GitHub token that can read this repository as the Colab
# secret GITHUB_TOKEN (key icon on the left). This cell mounts Drive, fetches
# the code and installs what Colab lacks. On this computer it uses DRIVE_DIR
# from .env and the `chama` conda environment.
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "m2-features-models"

if IN_COLAB:
    from google.colab import drive, userdata

    REPO = "/content/chama-loan-default"
    OWNER = "muriithi-alvin"
    PLAIN_URL = f"https://github.com/{OWNER}/chama-loan-default.git"

    def git(*args):
        """Run git with the token; show git's message, never the token."""
        token = userdata.get("GITHUB_TOKEN").strip()
        url = PLAIN_URL.replace("https://", f"https://{OWNER}:{token}@")
        result = subprocess.run(["git", *[url if a == "URL" else a for a in args]],
                                capture_output=True, text=True)
        if result.returncode:
            message = (result.stderr or result.stdout).replace(token, "***").strip()
            raise RuntimeError(f"git failed: {message}")

    drive.mount("/content/drive")
    if os.path.exists(REPO):
        git("-C", REPO, "pull", "-q", "URL", BRANCH)
    else:
        git("clone", "-q", "--branch", BRANCH, "URL", REPO)
        git("-C", REPO, "remote", "set-url", "origin", PLAIN_URL)   # no token left behind
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mlflow", "shap", "imbalanced-learn", "python-dotenv"], check=True)
    sys.path.insert(0, f"{REPO}/src")
    os.environ["DRIVE_DIR"] = "/content/drive/MyDrive/chama-loan-default"


import warnings

import pandas as pd

from chama.config import export_dir, load_config, mlflow_dir
from chama.models.data import TARGETS, known_outcomes, load_table, xy
from chama.models.train import target_settings

# the plots are matplotlib Figures built without pyplot: show them as images
%matplotlib inline
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
CONFIG = load_config()
DATA_DIR = export_dir()
print("data:  ", DATA_DIR)
print("MLflow:", mlflow_dir())

In [ ]:
# Which target: "loan_default" (will this loan default?) or
# "missed_repayment" (will a repayment be missed in the next meetings?)
TARGET = "loan_default"

target = TARGETS[TARGET]
SETTINGS = target_settings(CONFIG, TARGET)
DATA = DATA_DIR / f"{target.table}.parquet"
print(TARGET, "| label:", target.label, "| table:", DATA.name)

In [ ]:
from chama.models.train import setup_mlflow

mlflow = setup_mlflow(mlflow_dir(), SETTINGS["experiment"])
runs = mlflow.search_runs(order_by=["metrics.forward_pr_auc DESC"])
# runs from before batches were tagged have no training_batch
latest = runs[runs["tags.training_batch"] == runs["tags.training_batch"].dropna().max()]
latest[["tags.mlflow.runName", "metrics.forward_pr_auc", "metrics.forward_roc_auc",
        "metrics.forward_brier", "metrics.members_roc_auc", "tags.training_batch"]]

In [ ]:
# The run to explain: the best walk-forward PR AUC of the latest batch,
# or set RUN_NAME to another one from the table above.
RUN_NAME = None
chosen = (latest.iloc[0] if RUN_NAME is None
          else latest[latest["tags.mlflow.runName"] == RUN_NAME].iloc[0])
model = mlflow.sklearn.load_model(f"runs:/{chosen['run_id']}/model")
print("explaining:", chosen["tags.mlflow.runName"])

In [ ]:
from chama.models import plots
from chama.models.explain import importance, shap_values

df = load_table(DATA, target)
X_known, _ = xy(known_outcomes(df), target)
contributions = shap_values(model, X_known, background=X_known)
plots.importance_bar(importance(contributions), f"{TARGET}: {chosen['tags.mlflow.runName']}")

## What is still open

For `loan_default`, the loans whose outcome is not known yet; for
`missed_repayment`, the members owing at the latest meeting. Each gets a score, a
risk band and its three strongest factors.

In [ ]:
from IPython.display import display

from chama.models.evaluate import risk_band
from chama.models.explain import top_factors

open_rows = df[~df["in_training"]]
if TARGET == "missed_repayment":
    open_rows = open_rows[open_rows[target.time] == open_rows[target.time].max()]
X_open, _ = xy(open_rows, target)
scored = open_rows[list(target.keys)].assign(score=model.predict_proba(X_open)[:, 1].round(3))
scored["band"] = risk_band(scored["score"], SETTINGS["risk_bands"])
scored.sort_values("score", ascending=False)

In [ ]:
open_contributions = shap_values(model, X_open, background=X_known)
for index, row in open_contributions.iterrows():
    print(scored.loc[index, target.keys[0]], "| score", scored.loc[index, "score"],
          "|", scored.loc[index, "band"])
    display(top_factors(row))